In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/king_county_with_geo_features.csv")

df.head()

,id,price,bedrooms,bathrooms,sqft_living,zipcode,lat,long,dist_city_center_km
0,1,450000,3,2,1800,98103,47.6700,-122.3430,7.141089
1,2,620000,4,3,2400,98052,47.6740,-122.1215,17.487042
2,3,395000,2,1,1200,98115,47.6840,-122.3045,8.894623
3,4,810000,5,3,3200,98004,47.6101,-122.2015,9.800331
4,5,520000,3,2,1900,98105,47.6613,-122.3131,6.290075


In [3]:
parks = [
    (47.6038, -122.3301),
    (47.6687, -122.3762)
]

transit_stops = [
    (47.6097, -122.3331),
    (47.6735, -122.1214)
]

In [4]:
def haversine_distance(lat1, lon1, lat2, lon2):

    R = 6371

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat/2)**2 +
        np.cos(lat1) *
        np.cos(lat2) *
        np.sin(dlon/2)**2
    )

    c = 2 * np.arcsin(np.sqrt(a))

    return R * c

In [5]:
def nearest_distance(lat, lon, locations):

    distances = [
        haversine_distance(lat, lon, p_lat, p_lon)
        for p_lat, p_lon in locations
    ]

    return min(distances)

In [6]:
df["dist_to_nearest_park"] = df.apply(
    lambda row: nearest_distance(
        row["lat"],
        row["long"],
        parks
    ),
    axis=1
)

In [7]:
df["dist_to_transit_stop"] = df.apply(
    lambda row: nearest_distance(
        row["lat"],
        row["long"],
        transit_stops
    ),
    axis=1
)

In [8]:
df[
    [
        "id",
        "dist_to_nearest_park",
        "dist_to_transit_stop"
    ]
]

,id,dist_to_nearest_park,dist_to_transit_stop
0,1,2.490201,6.745955
1,2,17.469882,0.056099
2,3,5.631282,8.535061
3,4,9.666435,9.258039
4,5,4.796402,5.930127


In [10]:
df.to_csv(
    "../data/king_county_with_poi_features.csv",
    index=False
)